In [1]:
# import libaries
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# change the current working dir
os.chdir('/home/saad-naeem/Desktop/Level_4/Graduation_project_saad/chatbot/Baytology')

In [3]:
# read dataset as dataframe
df = pd.read_csv('../Baytology/DataSet/egypt_real_estate_listings.csv')

In [4]:
# explore data columns
print(f"df_columns: {df.columns}")

# insight -> we have 11 columns

df_columns: Index(['url', 'price', 'description', 'location', 'type', 'size', 'bedrooms',
       'bathrooms', 'available_from', 'payment_method', 'down_payment'],
      dtype='object')


In [5]:
# data describtion
df.info()

# insight -> we should convert every column to spesific datatype

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19924 entries, 0 to 19923
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   url             19924 non-null  object
 1   price           19385 non-null  object
 2   description     19846 non-null  object
 3   location        19833 non-null  object
 4   type            19847 non-null  object
 5   size            19847 non-null  object
 6   bedrooms        19780 non-null  object
 7   bathrooms       19784 non-null  object
 8   available_from  19261 non-null  object
 9   payment_method  19383 non-null  object
 10  down_payment    5445 non-null   object
dtypes: object(11)
memory usage: 1.7+ MB


In [6]:
# see the null values
df.isna().sum()

# insight -> we have to fill this null values

url                   0
price               539
description          78
location             91
type                 77
size                 77
bedrooms            144
bathrooms           140
available_from      663
payment_method      541
down_payment      14479
dtype: int64

In [7]:
f = df['down_payment'].isna().sum() 
print(f)

14479


In [8]:
# explore 'available_from' column
df['available_from']

0        31 Aug 2025
1         2 Sep 2025
2        19 Aug 2025
3        26 Aug 2025
4         2 Sep 2025
            ...     
19919    21 Aug 2025
19920     1 Sep 2025
19921    30 Jul 2025
19922    23 Aug 2025
19923    21 Aug 2025
Name: available_from, Length: 19924, dtype: object

# Data Cleaning

In [9]:
# Clean 'size' (Extract sqm numbers)
# This regex finds digits before " sqm"
df['size_sqm'] = df['size'].astype(str).str.extract(r'([\d,]+)\s*sqm')
df['size_sqm'] = pd.to_numeric(df['size_sqm'].str.replace(',', ''), errors='coerce')

# Fallback: If sqm is missing, try to find sqft and convert
mask_missing = df['size_sqm'].isna()
df.loc[mask_missing, 'temp_sqft'] = pd.to_numeric(
    df.loc[mask_missing, 'size'].astype(str).str.extract(r'([\d,]+)\s*sqft')[0].str.replace(',', ''), 
    errors='coerce'
)
df['size_sqm'] = df['size_sqm'].fillna(df['temp_sqft'] / 10.764)

In [10]:
# # Fallback: If sqm is missing, try to find sqft and convert
# mask_missing = df['size_sqm'].isna()
# df.loc[mask_missing, 'temp_sqft'] = pd.to_numeric(
#     df.loc[mask_missing, 'size'].astype(str).str.extract(r'([\d,]+)\s*sqft')[0].str.replace(',', ''), 
#     errors='coerce'
# )
# df['size_sqm'] = df['size_sqm'].fillna(df['temp_sqft'] / 10.764)

In [11]:
# Clean 'price' (remove commas, convert to number)
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(',', '', regex=False), errors='coerce')

In [12]:
# Clean 'down_payment'
df['down_payment'] = pd.to_numeric(df['down_payment'].astype(str).str.replace(',', '', regex=False), errors='coerce')

In [13]:
# Clean 'bedrooms' and 'bathrooms' (extract first number found)
df['bedrooms'] = pd.to_numeric(df['bedrooms'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
df['bathrooms'] = pd.to_numeric(df['bathrooms'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')

In [14]:
# see the null values
df.isna().sum()

# insight -> we have to fill this null values

url                   0
price               539
description          78
location             91
type                 77
size                 77
bedrooms            472
bathrooms           145
available_from      663
payment_method      541
down_payment      19924
size_sqm             77
temp_sqft         19924
dtype: int64

# --- Feature Engineering & Encoding ---


In [15]:
# Simplify 'location': Keep top 10, mark rest as 'Other'
top_10_locs = df['location'].value_counts().nlargest(10).index
df['location'] = df['location'].where(df['location'].isin(top_10_locs), 'Other')

In [16]:
# Drop columns not useful for the model
df_clean = df.drop(columns=['url', 'description', 'available_from', 'size', 'temp_sqft','down_payment','temp_sqft'], errors='ignore')
df_clean.to_csv('egypt_real_estate_preprocessed.csv', index=False)

In [17]:
# One-Hot Encode Categorical Variables (Type, Location, Payment Method)
df_encoded = pd.get_dummies(df_clean, columns=['location', 'type', 'payment_method'], drop_first=True)

# --- Handle Missing Values ---

In [18]:
# Decision Trees in sklearn cannot handle NaNs, so we fill them.
numerical_cols = ['price', 'size_sqm', 'bedrooms', 'bathrooms', 'down_payment']
for col in numerical_cols:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

In [19]:
# Final Check
print(f"Final shape: {df_encoded.shape}")
print(f"Remaining missing values: {df_encoded.isnull().sum().sum()}")

Final shape: (19924, 31)
Remaining missing values: 0


In [20]:
# Save
# df_encoded.to_csv('egypt_real_estate_preprocessed.csv', index=False)

# see data STD 